<a href="https://colab.research.google.com/github/AKSHAYAVISWA/HexaTraining/blob/main/June%2011/%20PySpark_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

spark = SparkSession.builder.appName("HospitalAnalytics").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [2]:
patients_data = """patient_id,patient_name,city,age,gender,blood_group,insurance_status
101,Rahul Sharma,Hyderabad,35,Male,O+,Active
102,Priya Reddy,Bangalore,29,Female,A+,Active
103,Amit Kumar,Mumbai,42,Male,B+,Inactive
104,Sneha Patel,Chennai,31,Female,O+,Active
105,Farhan Ali,Delhi,55,Male,AB+,Active
106,Neha Singh,,38,Female,A+,Inactive
107,Arjun Verma,Pune,26,Male,B+,Active
108,Meera Nair,Kochi,48,Female,O-,Active
109,Kiran Rao,Hyderabad,33,Male,,Inactive
110,Nisha Reddy,Bangalore,41,Female,A+,Active"""


appointments_data = """appointment_id,patient_id,doctor_name,department,appointment_date,consultation_fee,status
5001,101,Dr. Ramesh,Cardiology,2025-01-10,1500,Completed
5002,102,Dr. Suresh,Neurology,2025-01-11,2000,Completed
5003,101,Dr. Anita,Dermatology,2025-01-15,1000,Completed
5004,103,Dr. Ramesh,Cardiology,2025-01-20,1500,Cancelled
5005,104,Dr. Priya,Orthopedics,2025-01-22,2500,Completed
5006,105,Dr. Anita,Dermatology,2025-01-25,1000,Pending
5007,107,Dr. Suresh,Neurology,2025-02-01,2000,Completed
5008,110,Dr. Priya,Orthopedics,2025-02-03,2500,Completed
5009,120,Dr. Ramesh,Cardiology,2025-02-05,1500,Completed
5010,108,Dr. Anita,Dermatology,2025-02-10,,Pending"""

In [3]:
with open("patients.csv", "w") as f:
    f.write(patients_data.strip())

with open("appointments.csv", "w") as f:
    f.write(appointments_data.strip())

import json
preferences = [
    {"patient_id": 101, "preferred_hospital": "Apollo",
     "contact": {"phone": "9876500011", "email": "rahul@gmail.com"}},
    {"patient_id": 102, "preferred_hospital": "Yashoda",
     "contact": {"phone": None, "email": "priya@gmail.com"}},
    {"patient_id": 103, "preferred_hospital": "Care",
     "contact": {"phone": "9876500013", "email": None}},
    {"patient_id": 104, "preferred_hospital": None,
     "contact": {"phone": "9876500014", "email": "sneha@gmail.com"}}
]
with open("patient_preferences.json", "w") as f:
    json.dump(preferences, f)

In [4]:
patients_df = spark.read.csv("patients.csv", header=True, inferSchema=True)

In [5]:
appointments_df = spark.read.csv("appointments.csv", header=True, inferSchema=True)

In [7]:
patients_df.printSchema()
appointments_df.printSchema()

root
 |-- patient_id: integer (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- blood_group: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- appointment_id: integer (nullable = true)
 |-- patient_id: integer (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- status: string (nullable = true)



In [8]:
patients_df.show(5)
appointments_df.show(5)

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
+----------+------------+---------+---+------+-----------+----------------+
only showing top 5 rows
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|   

In [9]:
patients_df.select("city").distinct().show()

+---------+
|     city|
+---------+
|Bangalore|
|    Kochi|
|  Chennai|
|   Mumbai|
|     Pune|
|    Delhi|
|Hyderabad|
|     NULL|
+---------+



In [10]:
appointments_df.select("department").distinct().show()

+-----------+
| department|
+-----------+
|  Neurology|
|Dermatology|
| Cardiology|
|Orthopedics|
+-----------+



In [11]:
patients_df.write.mode("overwrite").parquet("patients_parquet")

In [12]:
parquet_df = spark.read.parquet("patients_parquet")
parquet_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [13]:
print("CSV Count:", patients_df.count())
print("Parquet Count:", parquet_df.count())

CSV Count: 10
Parquet Count: 10


In [16]:
from pyspark.sql.functions import col
patients_df.filter(col("city") == "Hyderabad").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [17]:
patients_df.filter(col("gender") == "Female").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [18]:
patients_df.filter(col("age") > 40).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [19]:
appointments_df.filter(col("status") == "Completed").show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
|          5009|       120| Dr. Ramesh| Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+-

In [20]:
appointments_df.filter(col("status") == "Pending").show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|Pending|
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [21]:
appointments_df.filter(col("consultation_fee") > 1500).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|Orthopedics|      2025-02-03|            2500|Completed|
+--------------+----------+-----------+-----------+----------------+----------------+---------+



In [22]:
patients_df.filter(col("insurance_status") == "Active").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [23]:
patients_df.filter(col("insurance_status") == "Inactive").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       106|  Neha Singh|     NULL| 38|Female|         A+|        Inactive|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [24]:
patients_df.filter(col("blood_group") == "O+").show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
+----------+------------+---------+---+------+-----------+----------------+



In [25]:
appointments_df.filter(col("department") == "Cardiology").show()

+--------------+----------+-----------+----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name|department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh|Cardiology|      2025-01-10|            1500|Completed|
|          5004|       103| Dr. Ramesh|Cardiology|      2025-01-20|            1500|Cancelled|
|          5009|       120| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+--------------+----------+-----------+----------+----------------+----------------+---------+



In [26]:
patients_df.filter(col("city").isNull()).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|       106|  Neha Singh|NULL| 38|Female|         A+|        Inactive|
+----------+------------+----+---+------+-----------+----------------+



In [27]:
patients_df.filter(col("blood_group").isNull()).show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
+----------+------------+---------+---+------+-----------+----------------+



In [28]:
appointments_df.filter(col("consultation_fee").isNull()).show()

+--------------+----------+-----------+-----------+----------------+----------------+-------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee| status|
+--------------+----------+-----------+-----------+----------------+----------------+-------+
|          5010|       108|  Dr. Anita|Dermatology|      2025-02-10|            NULL|Pending|
+--------------+----------+-----------+-----------+----------------+----------------+-------+



In [29]:
from pyspark.sql.functions import col, sum

patients_df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in patients_df.columns
]).show()

+----------+------------+----+---+------+-----------+----------------+
|patient_id|patient_name|city|age|gender|blood_group|insurance_status|
+----------+------------+----+---+------+-----------+----------------+
|         0|           0|   1|  0|     0|          1|               0|
+----------+------------+----+---+------+-----------+----------------+



In [30]:
patients_df = patients_df.fillna({"city": "Unknown"})
patients_df.show()

+----------+------------+---------+---+------+-----------+----------------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|
+----------+------------+---------+---+------+-----------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|         B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|         A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|         B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|         O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|       NULL|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|
+----------+

In [31]:
patients_df = patients_df.fillna({"blood_group": "Not Available"})
patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|
+----------+------------+---------+---+------+-------------+----------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|
|       108|  Meera Nair|    Kochi| 48|Female|           O-|          Active|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|
|       110| Nisha Reddy|Bangalore| 41|Female|           A+|    

In [32]:
appointments_df = appointments_df.fillna({"consultation_fee": 0})
appointments_df.show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [33]:
appointments_df.dropna(subset=["consultation_fee"]).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|
+--------------+----------+-----------+-----------+----------------+----------------+---------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|
|          5007|       107| Dr. Suresh|  Neurology|      2025-02-01|            2000|Completed|
|          5008|       110|  Dr. Priya|O

In [34]:
from pyspark.sql.functions import when

patients_df = patients_df.withColumn(
    "data_quality_status",
    when(
        col("city").isNull() | col("blood_group").isNull(),
        "Incomplete"
    ).otherwise("Complete")
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|
|       107| Arjun Verma|     Pune| 26|  Male|           B+|          Active|           Complete|
|       108|  Meera 

In [35]:
patients_df.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   10|
+-------------------+-----+



In [36]:
from pyspark.sql.functions import upper

patients_df.select(
    "patient_name",
    upper("patient_name").alias("upper_name")
).show()

+------------+------------+
|patient_name|  upper_name|
+------------+------------+
|Rahul Sharma|RAHUL SHARMA|
| Priya Reddy| PRIYA REDDY|
|  Amit Kumar|  AMIT KUMAR|
| Sneha Patel| SNEHA PATEL|
|  Farhan Ali|  FARHAN ALI|
|  Neha Singh|  NEHA SINGH|
| Arjun Verma| ARJUN VERMA|
|  Meera Nair|  MEERA NAIR|
|   Kiran Rao|   KIRAN RAO|
| Nisha Reddy| NISHA REDDY|
+------------+------------+



In [37]:
from pyspark.sql.functions import lower

patients_df.select(
    "patient_name",
    lower("patient_name").alias("lower_name")
).show()

+------------+------------+
|patient_name|  lower_name|
+------------+------------+
|Rahul Sharma|rahul sharma|
| Priya Reddy| priya reddy|
|  Amit Kumar|  amit kumar|
| Sneha Patel| sneha patel|
|  Farhan Ali|  farhan ali|
|  Neha Singh|  neha singh|
| Arjun Verma| arjun verma|
|  Meera Nair|  meera nair|
|   Kiran Rao|   kiran rao|
| Nisha Reddy| nisha reddy|
+------------+------------+



In [38]:
from pyspark.sql.functions import length

patients_df.select(
    "patient_name",
    length("patient_name").alias("name_length")
).show()

+------------+-----------+
|patient_name|name_length|
+------------+-----------+
|Rahul Sharma|         12|
| Priya Reddy|         11|
|  Amit Kumar|         10|
| Sneha Patel|         11|
|  Farhan Ali|         10|
|  Neha Singh|         10|
| Arjun Verma|         11|
|  Meera Nair|         10|
|   Kiran Rao|          9|
| Nisha Reddy|         11|
+------------+-----------+



In [39]:
from pyspark.sql.functions import substring

patients_df.select(
    "patient_name",
    substring("patient_name", 1, 3).alias("first_3_letters")
).show()

+------------+---------------+
|patient_name|first_3_letters|
+------------+---------------+
|Rahul Sharma|            Rah|
| Priya Reddy|            Pri|
|  Amit Kumar|            Ami|
| Sneha Patel|            Sne|
|  Farhan Ali|            Far|
|  Neha Singh|            Neh|
| Arjun Verma|            Arj|
|  Meera Nair|            Mee|
|   Kiran Rao|            Kir|
| Nisha Reddy|            Nis|
+------------+---------------+



In [40]:
patients_df = patients_df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Middle")
    .otherwise("Senior")
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|   Middle|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|   Middle|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|   Senior|
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|   Middle|
|       107| Arjun Verma|   

In [41]:
patients_df = patients_df.withColumn(
    "insurance_flag",
    when(col("insurance_status") == "Active", 1)
    .otherwise(0)
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|   Middle|             0|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|   Middle|             1|
|       105|  Farhan Ali|    Delhi| 55|  Male|          AB+|          Active|           Complete|   Senior|             1|
|       106|  Ne

In [42]:
patients_df = patients_df.withColumn(
    "senior_citizen",
    when(col("age") >= 60, "Yes")
    .otherwise("No")
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|   Middle|             0|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|   Middle|             1|            No|
|       105|  Farhan Ali|    Delhi

In [43]:
from pyspark.sql.functions import concat_ws

patients_df.select(
    concat_ws(" - ", "patient_name", "city")
    .alias("patient_city")
).show()

+--------------------+
|        patient_city|
+--------------------+
|Rahul Sharma - Hy...|
|Priya Reddy - Ban...|
| Amit Kumar - Mumbai|
|Sneha Patel - Che...|
|  Farhan Ali - Delhi|
|Neha Singh - Unknown|
|  Arjun Verma - Pune|
|  Meera Nair - Kochi|
|Kiran Rao - Hyder...|
|Nisha Reddy - Ban...|
+--------------------+



In [44]:
from pyspark.sql.functions import trim

patients_df = patients_df.withColumn(
    "patient_name",
    trim(col("patient_name"))
)

patients_df.show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|   Middle|             0|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|   Middle|             1|            No|
|       105|  Farhan Ali|    Delhi

In [45]:
patients_df.select(
    upper("city").alias("city_uppercase")
).show()

+--------------+
|city_uppercase|
+--------------+
|     HYDERABAD|
|     BANGALORE|
|        MUMBAI|
|       CHENNAI|
|         DELHI|
|       UNKNOWN|
|          PUNE|
|         KOCHI|
|     HYDERABAD|
|     BANGALORE|
+--------------+



In [46]:
patients_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    1|
|  Unknown|    1|
|     Pune|    1|
|    Delhi|    1|
|Hyderabad|    2|
+---------+-----+



In [47]:
patients_df.groupBy("gender").count().show()

+------+-----+
|gender|count|
+------+-----+
|Female|    5|
|  Male|    5|
+------+-----+



In [48]:
patients_df.groupBy("blood_group").count().show()

+-------------+-----+
|  blood_group|count|
+-------------+-----+
|          AB+|    1|
|           O+|    2|
|           O-|    1|
|           B+|    2|
|           A+|    3|
|Not Available|    1|
+-------------+-----+



In [49]:
appointments_df.groupBy("department").count().show()

+-----------+-----+
| department|count|
+-----------+-----+
|  Neurology|    2|
|Dermatology|    3|
| Cardiology|    3|
|Orthopedics|    2|
+-----------+-----+



In [50]:
from pyspark.sql.functions import avg

patients_df.groupBy("city") \
           .agg(avg("age").alias("average_age")) \
           .show()

+---------+-----------+
|     city|average_age|
+---------+-----------+
|Bangalore|       35.0|
|    Kochi|       48.0|
|  Chennai|       31.0|
|   Mumbai|       42.0|
|  Unknown|       38.0|
|     Pune|       26.0|
|    Delhi|       55.0|
|Hyderabad|       34.0|
+---------+-----------+



In [51]:
from pyspark.sql.functions import max

patients_df.groupBy("city") \
           .agg(max("age").alias("max_age")) \
           .show()

+---------+-------+
|     city|max_age|
+---------+-------+
|Bangalore|     41|
|    Kochi|     48|
|  Chennai|     31|
|   Mumbai|     42|
|  Unknown|     38|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     35|
+---------+-------+



In [52]:
from pyspark.sql.functions import min

patients_df.groupBy("city") \
           .agg(min("age").alias("min_age")) \
           .show()

+---------+-------+
|     city|min_age|
+---------+-------+
|Bangalore|     29|
|    Kochi|     48|
|  Chennai|     31|
|   Mumbai|     42|
|  Unknown|     38|
|     Pune|     26|
|    Delhi|     55|
|Hyderabad|     33|
+---------+-------+



In [53]:
appointments_df.groupBy("department") \
               .agg(avg("consultation_fee").alias("avg_fee")) \
               .show()

+-----------+-----------------+
| department|          avg_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [54]:
from pyspark.sql.functions import sum

appointments_df.groupBy("department") \
               .agg(sum("consultation_fee").alias("total_fee")) \
               .show()

+-----------+---------+
| department|total_fee|
+-----------+---------+
|  Neurology|     4000|
|Dermatology|     2000|
| Cardiology|     4500|
|Orthopedics|     5000|
+-----------+---------+



In [55]:
from pyspark.sql.functions import sum

appointments_df.groupBy("department") \
               .agg(sum("consultation_fee").alias("revenue")) \
               .orderBy(col("revenue").desc()) \
               .show(1)

+-----------+-------+
| department|revenue|
+-----------+-------+
|Orthopedics|   5000|
+-----------+-------+
only showing top 1 row


In [56]:
patients_df.join(
    appointments_df,
    "patient_id",
    "inner"
).show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|   Middle|             1|            No|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|           Complete|   Middle|             1|    

In [57]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|         

In [58]:
patients_df.join(
    appointments_df,
    "patient_id",
    "right"
).show()

+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-----------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|         O+|          Active|           Complete|   Middle|             1|            No|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       102| Priya Reddy|Bangalore|  29|Female|         A+|          Active|           Complete|    Young|             1

In [59]:
patients_df.join(
    appointments_df,
    "patient_id",
    "full"
).show()

+----------+------------+---------+----+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|patient_id|patient_name|     city| age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|
+----------+------------+---------+----+------+-------------+----------------+-------------------+---------+--------------+--------------+--------------+-----------+-----------+----------------+----------------+---------+
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|
|       101|Rahul Sharma|Hyderabad|  35|  Male|           O+|          Active|           Complete|   Middle|    

In [60]:
patients_df.join(
    appointments_df,
    "patient_id",
    "left_anti"
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       106|  Neha Singh|  Unknown| 38|Female|           A+|        Inactive|           Complete|   Middle|             0|            No|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|           Complete|   Middle|             0|            No|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+



In [61]:
appointments_df.join(
    patients_df,
    "patient_id",
    "left_anti"
).show()

+----------+--------------+-----------+----------+----------------+----------------+---------+
|patient_id|appointment_id|doctor_name|department|appointment_date|consultation_fee|   status|
+----------+--------------+-----------+----------+----------------+----------------+---------+
|       120|          5009| Dr. Ramesh|Cardiology|      2025-02-05|            1500|Completed|
+----------+--------------+-----------+----------+----------------+----------------+---------+



In [62]:
appointments_df.groupBy("patient_id") \
               .count() \
               .show()

+----------+-----+
|patient_id|count|
+----------+-----+
|       108|    1|
|       101|    2|
|       103|    1|
|       120|    1|
|       107|    1|
|       102|    1|
|       105|    1|
|       110|    1|
|       104|    1|
+----------+-----+



In [63]:
appointments_df.groupBy("patient_id") \
               .agg(sum("consultation_fee")
               .alias("total_spent")) \
               .show()

+----------+-----------+
|patient_id|total_spent|
+----------+-----------+
|       108|          0|
|       101|       2500|
|       103|       1500|
|       120|       1500|
|       107|       2000|
|       102|       2000|
|       105|       1000|
|       110|       2500|
|       104|       2500|
+----------+-----------+



In [64]:
appointments_df.groupBy("patient_id") \
               .agg(sum("consultation_fee")
               .alias("total_spent")) \
               .orderBy(col("total_spent").desc()) \
               .show(1)

+----------+-----------+
|patient_id|total_spent|
+----------+-----------+
|       110|       2500|
+----------+-----------+
only showing top 1 row


In [65]:
appointments_df.groupBy("patient_id") \
               .count() \
               .withColumnRenamed("count",
                                  "appointment_count") \
               .show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [66]:
appointments_df.groupBy("patient_id") \
               .count() \
               .withColumnRenamed("count",
                                  "appointment_count") \
               .show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [68]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
patient_spending = appointments_df.groupBy("patient_id") \
    .agg(sum("consultation_fee").alias("total_spent"))

windowSpec = Window.orderBy(col("total_spent").desc())

patient_spending.withColumn(
    "rank",
    rank().over(windowSpec)
).show()

+----------+-----------+----+
|patient_id|total_spent|rank|
+----------+-----------+----+
|       101|       2500|   1|
|       110|       2500|   1|
|       104|       2500|   1|
|       107|       2000|   4|
|       102|       2000|   4|
|       103|       1500|   6|
|       120|       1500|   6|
|       105|       1000|   8|
|       108|          0|   9|
+----------+-----------+----+



In [69]:
patient_spending.withColumn(
    "row_number",
    row_number().over(windowSpec)
).show()

+----------+-----------+----------+
|patient_id|total_spent|row_number|
+----------+-----------+----------+
|       101|       2500|         1|
|       110|       2500|         2|
|       104|       2500|         3|
|       107|       2000|         4|
|       102|       2000|         5|
|       103|       1500|         6|
|       120|       1500|         7|
|       105|       1000|         8|
|       108|          0|         9|
+----------+-----------+----------+



In [70]:
patient_spending.orderBy(
    col("total_spent").desc()
).show(1)

+----------+-----------+
|patient_id|total_spent|
+----------+-----------+
|       110|       2500|
+----------+-----------+
only showing top 1 row


In [71]:
patient_spending.orderBy(
    col("total_spent").desc()
).show(3)

+----------+-----------+
|patient_id|total_spent|
+----------+-----------+
|       101|       2500|
|       110|       2500|
|       104|       2500|
+----------+-----------+
only showing top 3 rows


In [72]:
patient_city_spending = patients_df.join(
    patient_spending,
    "patient_id"
)

cityWindow = Window.partitionBy("city") \
                   .orderBy(col("total_spent").desc())

patient_city_spending.withColumn(
    "rank",
    rank().over(cityWindow)
).filter(col("rank") == 1).show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+-----------+----+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|total_spent|rank|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+-----------+----+
|       110| Nisha Reddy|Bangalore| 41|Female|         A+|          Active|           Complete|   Middle|             1|            No|       2500|   1|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|   Middle|             1|            No|       2500|   1|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|   Senior|             1|            No|       1000|   1|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|       

In [73]:
cityWindow = Window.partitionBy("city") \
                   .orderBy(col("total_spent").asc())

patient_city_spending.withColumn(
    "rank",
    rank().over(cityWindow)
).filter(col("rank") == 1).show()

+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+-----------+----+
|patient_id|patient_name|     city|age|gender|blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|total_spent|rank|
+----------+------------+---------+---+------+-----------+----------------+-------------------+---------+--------------+--------------+-----------+----+
|       102| Priya Reddy|Bangalore| 29|Female|         A+|          Active|           Complete|    Young|             1|            No|       2000|   1|
|       104| Sneha Patel|  Chennai| 31|Female|         O+|          Active|           Complete|   Middle|             1|            No|       2500|   1|
|       105|  Farhan Ali|    Delhi| 55|  Male|        AB+|          Active|           Complete|   Senior|             1|            No|       1000|   1|
|       101|Rahul Sharma|Hyderabad| 35|  Male|         O+|          Active|       

In [74]:
runningWindow = Window.orderBy("appointment_id")

appointments_df.withColumn(
    "running_total",
    sum("consultation_fee").over(runningWindow)
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+-------------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|running_total|
+--------------+----------+-----------+-----------+----------------+----------------+---------+-------------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|         1500|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|         3500|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|         4500|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|         6000|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|         8500|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|         9500|
|         

In [75]:
leadWindow = Window.orderBy("appointment_id")

appointments_df.withColumn(
    "next_fee",
    lead("consultation_fee").over(leadWindow)
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|next_fee|
+--------------+----------+-----------+-----------+----------------+----------------+---------+--------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|    2000|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|    1000|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|    1500|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|    2500|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|    1000|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|    2000|
|          5007|       107| Dr. Suresh|  Neurology|    

In [76]:
appointments_df.withColumn(
    "previous_fee",
    lag("consultation_fee").over(leadWindow)
).show()

+--------------+----------+-----------+-----------+----------------+----------------+---------+------------+
|appointment_id|patient_id|doctor_name| department|appointment_date|consultation_fee|   status|previous_fee|
+--------------+----------+-----------+-----------+----------------+----------------+---------+------------+
|          5001|       101| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|        NULL|
|          5002|       102| Dr. Suresh|  Neurology|      2025-01-11|            2000|Completed|        1500|
|          5003|       101|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|        2000|
|          5004|       103| Dr. Ramesh| Cardiology|      2025-01-20|            1500|Cancelled|        1000|
|          5005|       104|  Dr. Priya|Orthopedics|      2025-01-22|            2500|Completed|        1500|
|          5006|       105|  Dr. Anita|Dermatology|      2025-01-25|            1000|  Pending|        2500|
|          5007|   

In [77]:
preferences_df = spark.read.json(
    "patient_preferences.json"
)

preferences_df.show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{rahul@gmail.com,...|       101|            Apollo|
|{priya@gmail.com,...|       102|           Yashoda|
|  {NULL, 9876500013}|       103|              Care|
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [78]:
preferences_df.printSchema()

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: long (nullable = true)
 |-- preferred_hospital: string (nullable = true)



In [79]:
preferences_df.select(
    "patient_id",
    col("contact.phone").alias("phone")
).show()

+----------+----------+
|patient_id|     phone|
+----------+----------+
|       101|9876500011|
|       102|      NULL|
|       103|9876500013|
|       104|9876500014|
+----------+----------+



In [80]:
preferences_df.select(
    "patient_id",
    col("contact.email").alias("email")
).show()

+----------+---------------+
|patient_id|          email|
+----------+---------------+
|       101|rahul@gmail.com|
|       102|priya@gmail.com|
|       103|           NULL|
|       104|sneha@gmail.com|
+----------+---------------+



In [81]:
preferences_df.filter(
    col("contact.phone").isNull()
).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{priya@gmail.com,...|       102|           Yashoda|
+--------------------+----------+------------------+



In [82]:
preferences_df.filter(
    col("contact.email").isNull()
).show()

+------------------+----------+------------------+
|           contact|patient_id|preferred_hospital|
+------------------+----------+------------------+
|{NULL, 9876500013}|       103|              Care|
+------------------+----------+------------------+



In [83]:
preferences_df.filter(
    col("preferred_hospital").isNull()
).show()

+--------------------+----------+------------------+
|             contact|patient_id|preferred_hospital|
+--------------------+----------+------------------+
|{sneha@gmail.com,...|       104|              NULL|
+--------------------+----------+------------------+



In [86]:
from pyspark.sql.functions import col

pref_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

pref_flat = pref_flat.fillna({
    "phone": "Not Available",
    "email": "Not Available"
})


In [87]:
pref_flat.fillna({"email": "Not Available"}).show()

+----------+------------------+-------------+---------------+
|patient_id|preferred_hospital|        phone|          email|
+----------+------------------+-------------+---------------+
|       101|            Apollo|   9876500011|rahul@gmail.com|
|       102|           Yashoda|Not Available|priya@gmail.com|
|       103|              Care|   9876500013|  Not Available|
|       104|              NULL|   9876500014|sneha@gmail.com|
+----------+------------------+-------------+---------------+



In [88]:
patients_df.join(
    pref_flat,
    "patient_id",
    "left"
).show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+------------------+-------------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|preferred_hospital|        phone|          email|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+------------------+-------------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|            Apollo|   9876500011|rahul@gmail.com|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|           Yashoda|Not Available|priya@gmail.com|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|     

In [90]:
patients_df.createOrReplaceTempView("patients")

In [91]:
appointments_df.createOrReplaceTempView("appointments")

In [92]:
spark.sql("""
SELECT *
FROM patients
""").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|
|       102| Priya Reddy|Bangalore| 29|Female|           A+|          Active|           Complete|    Young|             1|            No|
|       103|  Amit Kumar|   Mumbai| 42|  Male|           B+|        Inactive|           Complete|   Middle|             0|            No|
|       104| Sneha Patel|  Chennai| 31|Female|           O+|          Active|           Complete|   Middle|             1|            No|
|       105|  Farhan Ali|    Delhi

In [93]:
spark.sql("""
SELECT *
FROM patients
WHERE city = 'Hyderabad'
""").show()

+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|data_quality_status|age_group|insurance_flag|senior_citizen|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|           Complete|   Middle|             1|            No|
|       109|   Kiran Rao|Hyderabad| 33|  Male|Not Available|        Inactive|           Complete|   Middle|             0|            No|
+----------+------------+---------+---+------+-------------+----------------+-------------------+---------+--------------+--------------+



In [94]:
spark.sql("""
SELECT city,
       COUNT(*) AS patient_count
FROM patients
GROUP BY city
""").show()

+---------+-------------+
|     city|patient_count|
+---------+-------------+
|Bangalore|            2|
|    Kochi|            1|
|  Chennai|            1|
|   Mumbai|            1|
|  Unknown|            1|
|     Pune|            1|
|    Delhi|            1|
|Hyderabad|            2|
+---------+-------------+



In [95]:
spark.sql("""
SELECT department,
       COUNT(*) AS appointment_count
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|appointment_count|
+-----------+-----------------+
|  Neurology|                2|
|Dermatology|                3|
| Cardiology|                3|
|Orthopedics|                2|
+-----------+-----------------+



In [96]:
spark.sql("""
SELECT department,
       AVG(consultation_fee) AS avg_fee
FROM appointments
GROUP BY department
""").show()

+-----------+-----------------+
| department|          avg_fee|
+-----------+-----------------+
|  Neurology|           2000.0|
|Dermatology|666.6666666666666|
| Cardiology|           1500.0|
|Orthopedics|           2500.0|
+-----------+-----------------+



In [97]:
spark.sql("""
SELECT MAX(consultation_fee) AS highest_fee
FROM appointments
""").show()

+-----------+
|highest_fee|
+-----------+
|       2500|
+-----------+



In [98]:
spark.sql("""
SELECT patient_id,
       COUNT(*) AS appointment_count
FROM appointments
GROUP BY patient_id
""").show()

+----------+-----------------+
|patient_id|appointment_count|
+----------+-----------------+
|       108|                1|
|       101|                2|
|       103|                1|
|       120|                1|
|       107|                1|
|       102|                1|
|       105|                1|
|       110|                1|
|       104|                1|
+----------+-----------------+



In [99]:
spark.sql("""
SELECT patient_id,
       SUM(consultation_fee) AS total_spent
FROM appointments
GROUP BY patient_id
ORDER BY total_spent DESC
LIMIT 5
""").show()


+----------+-----------+
|patient_id|total_spent|
+----------+-----------+
|       101|       2500|
|       110|       2500|
|       104|       2500|
|       107|       2000|
|       102|       2000|
+----------+-----------+



In [100]:
patients_df = spark.read.csv(
    "patients.csv",
    header=True,
    inferSchema=True
)

appointments_df = spark.read.csv(
    "appointments.csv",
    header=True,
    inferSchema=True
)

In [101]:
preferences_df = spark.read.json(
    "patient_preferences.json"
)

In [102]:
patients_df = patients_df.fillna({
    "city": "Unknown",
    "blood_group": "Not Available"
})

appointments_df = appointments_df.fillna({
    "consultation_fee": 0
})

In [103]:
from pyspark.sql.functions import col

pref_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

final_df = patients_df.join(
    appointments_df,
    "patient_id",
    "left"
).join(
    pref_flat,
    "patient_id",
    "left"
)

final_df.show()

+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+----------+---------------+
|patient_id|patient_name|     city|age|gender|  blood_group|insurance_status|appointment_id|doctor_name| department|appointment_date|consultation_fee|   status|preferred_hospital|     phone|          email|
+----------+------------+---------+---+------+-------------+----------------+--------------+-----------+-----------+----------------+----------------+---------+------------------+----------+---------------+
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5003|  Dr. Anita|Dermatology|      2025-01-15|            1000|Completed|            Apollo|9876500011|rahul@gmail.com|
|       101|Rahul Sharma|Hyderabad| 35|  Male|           O+|          Active|          5001| Dr. Ramesh| Cardiology|      2025-01-10|            1500|Completed|            

In [104]:
from pyspark.sql.functions import when

final_df = final_df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Middle")
    .otherwise("Senior")
)

In [105]:
department_revenue = final_df.groupBy(
    "department"
).agg(
    sum("consultation_fee").alias("department_revenue")
)

department_revenue.show()

+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|       NULL|              NULL|
|  Neurology|              4000|
|Dermatology|              2000|
| Cardiology|              3000|
|Orthopedics|              5000|
+-----------+------------------+



In [106]:
patient_spending = final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("consultation_fee").alias("total_spent")
)

patient_spending.show()

+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       107| Arjun Verma|       2000|
|       108|  Meera Nair|          0|
|       109|   Kiran Rao|       NULL|
|       110| Nisha Reddy|       2500|
|       105|  Farhan Ali|       1000|
|       101|Rahul Sharma|       2500|
|       104| Sneha Patel|       2500|
|       103|  Amit Kumar|       1500|
|       106|  Neha Singh|       NULL|
|       102| Priya Reddy|       2000|
+----------+------------+-----------+



In [107]:
final_df.groupBy("department") \
        .agg(sum("consultation_fee")
        .alias("total_revenue")) \
        .show()

+-----------+-------------+
| department|total_revenue|
+-----------+-------------+
|       NULL|         NULL|
|  Neurology|         4000|
|Dermatology|         2000|
| Cardiology|         3000|
|Orthopedics|         5000|
+-----------+-------------+



In [108]:
final_df.write.mode("overwrite") \
       .parquet("hospital_analytics_output")

In [109]:
print("Total Patients:",
      patients_df.count())

print("Total Appointments:",
      appointments_df.count())

patient_spending.orderBy(
    col("total_spent").desc()
).show(5)

department_revenue.orderBy(
    col("department_revenue").desc()
).show()

Total Patients: 10
Total Appointments: 10
+----------+------------+-----------+
|patient_id|patient_name|total_spent|
+----------+------------+-----------+
|       110| Nisha Reddy|       2500|
|       101|Rahul Sharma|       2500|
|       104| Sneha Patel|       2500|
|       107| Arjun Verma|       2000|
|       102| Priya Reddy|       2000|
+----------+------------+-----------+
only showing top 5 rows
+-----------+------------------+
| department|department_revenue|
+-----------+------------------+
|Orthopedics|              5000|
|  Neurology|              4000|
| Cardiology|              3000|
|Dermatology|              2000|
|       NULL|              NULL|
+-----------+------------------+

